# ML-03 — Frame Your Lane as an ML Task

**Lane: Refresh / Content Opportunity Scoring.** This is a provisional frame using the bundled starter slice; it is decision-support, not an instruction to change content.

## 1. My lane as an ML task (type)

**Task type: ranking / scoring.** I want one review-priority score per eligible content item, then a human editor can review the highest-ranked items first. The decision is *which item should receive limited editorial attention first?* A missed opportunity can leave a useful page unreviewed; a false alarm can waste editor time. The score supports review, not automatic editing.

In [1]:
import pandas as pd

pd.DataFrame([
    {"decision": "Which eligible content item should an editor review first?", "output": "A ranked review queue", "action": "Inspect the page and decide whether to refresh, expand, protect, or monitor it"}
])

,decision,output,action
0,Which eligible content item should an editor r...,A ranked review queue,Inspect the page and decide whether to refresh...


## 2. Target or proxy

The starter does not provide a later, post-review outcome. For an initial comparison only, I will use a **decline proxy**: `trend_direction == "down"`. In this dataset, that category is defined from a recent-versus-prior impressions rule (`trend_pct`), so it is not an independent future outcome. A high score that finds this proxy is only evidence of agreement with this present-period label—not evidence that a refresh will work or that the page will decline next month.

For a later, stronger version, I would define a scoring date and label a separately measured outcome in a subsequent time window. I must not use `trend_direction` or `trend_pct` as model inputs when predicting this proxy, because they define it.

In [2]:
from pathlib import Path
import sys

csv_name = "content_refresh_anonymized.csv"
data_candidates = [base / "data" / "raw" / csv_name for base in (Path.cwd(), *Path.cwd().parents)]
data_path = next((path for path in data_candidates if path.is_file()), None)

if data_path is None and "google.colab" in sys.modules:
    import subprocess

    repo_dir = Path.cwd() / "flyrank-ml-internship-starter"
    csv_path = repo_dir / "data" / "raw" / csv_name
    if not csv_path.is_file():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter", str(repo_dir)],
            check=True,
        )
    data_path = csv_path

if data_path is None:
    raise FileNotFoundError(f"Could not find data/raw/{csv_name} from the current directory or its parents.")

raw_df = pd.read_csv(data_path)
required_columns = {
    "content_id", "impressions_90d", "content_age_days", "trend_direction",
    "content_type", "days_since_last_update", "avg_position", "ctr", "engagement_rate",
}
missing_columns = sorted(required_columns.difference(raw_df.columns))
if missing_columns:
    raise ValueError(f"Starter CSV is missing required columns: {missing_columns}")

lane_df = raw_df.loc[
    (raw_df["impressions_90d"] > 0) & (raw_df["content_age_days"] >= 90)
].drop_duplicates(subset="content_id").copy()
lane_df["is_declining_proxy"] = lane_df["trend_direction"].str.lower().eq("down").astype("int8")
print(f"Loaded {len(raw_df):,} starter rows; {len(lane_df):,} eligible content items in this lane.")
print(f"Decline-proxy prevalence in the eligible slice: {lane_df['is_declining_proxy'].mean():.1%}.")

Loaded 30,000 starter rows; 30,000 eligible content items in this lane.
Decline-proxy prevalence in the eligible slice: 54.2%.


## 3. Success metric

I will use **Precision@50**: among the 50 highest-ranked eligible items, what fraction match the decline proxy? A useful next model should beat a simple, proxy-blind fixed-rule baseline (here, oldest update first) on a client-held-out evaluation. I will report the proxy's overall prevalence alongside the metric, and call the result useful only if the gain is repeatable and large enough to justify the review effort. Precision@50 measures agreement with this proxy—not future refresh success.

In [3]:
k = min(50, len(lane_df))
if k == 0:
    raise ValueError("No eligible content items are available to calculate Precision@50.")

oldest_first = lane_df.sort_values("days_since_last_update", ascending=False, na_position="last").head(k)
oldest_first_precision_at_k = oldest_first["is_declining_proxy"].mean()
print(f"Eligible-slice proxy prevalence: {lane_df['is_declining_proxy'].mean():.3f}")
print(f"Fixed baseline Precision@{k} (oldest update first): {oldest_first_precision_at_k:.3f}")

Eligible-slice proxy prevalence: 0.542
Fixed baseline Precision@50 (oldest update first): 0.520


## 4. The unit of analysis, as a real dataframe

**One row = one eligible pseudonymized content item** in the starter snapshot, with positive 90-day impressions and at least 90 days of content age; duplicate content IDs are removed to match the reference preparation step. The preview omits content and client IDs. The final column is the proxy label, not a causal outcome.

In [4]:
display_columns = [
    "content_type", "content_age_days", "days_since_last_update",
    "impressions_90d", "avg_position", "is_declining_proxy",
]
lane_df[display_columns].head(5)

,content_type,content_age_days,days_since_last_update,impressions_90d,avg_position,is_declining_proxy
0,keyword article,187,20,3803,10.6,1
1,keyword article,445,25,15320,20.3,1
2,keyword article,141,20,12581,36.5,1
3,keyword article,463,22,11751,6.2,0
4,keyword article,263,14,19140,44.0,1


## 5. Why ML beats a fixed rule here

A single cutoff such as “review the oldest pages” ignores whether an item has meaningful search demand, its visibility, engagement, and content type. These signals can interact, so a model may prioritize more effectively than one hand-written threshold. That is a hypothesis to test—not a guaranteed win. A transparent fixed rule remains the baseline, and if the model does not reliably improve Precision@50 on held-out clients, the simpler rule is preferable. Editors make the final call; this score does not establish that refreshing a page causes recovery.

In [5]:
candidate_signals = [
    "content_age_days", "days_since_last_update", "content_type",
    "impressions_90d", "avg_position", "ctr", "engagement_rate",
]
label_defining_columns = {"trend_direction", "trend_pct"}
assert not label_defining_columns.intersection(candidate_signals)
print("Candidate signals span content age, freshness, demand, visibility, engagement, and type.")
print("Leakage check passed: neither label-defining trend field is in the candidate signal list.")

Candidate signals span content age, freshness, demand, visibility, engagement, and type.
Leakage check passed: neither label-defining trend field is in the candidate signal list.


## Self-check

- [x] The task type, target/proxy, success metric, unit of analysis, action, and ML rationale are filled in.
- [x] The code loads the local starter slice and displays a real, ID-free dataframe preview.
- [ ] Run this notebook top to bottom in your own environment and review the outputs.
- [x] No client names, URLs, or private queries are included.
- [x] Claims are limited to proxy-based, decision-support findings; no causal claim is made.
- [ ] Commit the executed notebook to your own public repository, then submit that repository URL on the card.